# EEG_17 — Matrici di Connettività per Cluster Semantico

**Obiettivo**: Per ogni cluster semantico (concr4: 0,1,2,3), confrontare la matrice di connettività
media di ciascun soggetto con la matrice media globale dello stesso cluster.

**Logica**:
- Ogni trial appartiene a un cluster semantico (0=categoria A, 1=B, 2=C, 3=D)
- Per ogni soggetto × cluster: media delle matrici `adj` dei trial di quel cluster → `(61,61)`
- Grand mean cluster k: media su tutti i soggetti → `(61,61)`
- Diff: `conn_soggetto_cluster_k − grand_cluster_k`

**Risponde a**: quando il soggetto X immagina parole della categoria k,
la sua connettività si discosta dalla media della popolazione per la stessa categoria?

In [ ]:
import json, logging, math, re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-8s %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('eeg17')

project_root = next((p for p in [Path.cwd()] + list(Path.cwd().parents)
                     if (p / '.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'eeg17'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_CHANNELS     = 61
N_SAMPLES      = 384
CLUSTER_SCHEME = 'concr4'
N_SEM_CLUSTERS = 4      # cluster semantici concr4: 0,1,2,3

METRICS  = ['pcc', 'abs_pcc', 'im_pcc', 'wpli', 'plv']
PRIMARY  = 'abs_pcc'
RANDOM_SEED = 42

# Accuracy post-hoc
ACC_FILE_12 = project_root / 'figures' / 'eeg12_subject_ranking.csv'
ACC_FILE_13 = project_root / 'figures' / 'eeg13b_subject_ranking.csv'

# Mapping word → cluster semantico
label2cluster = {int(k): int(v) for k, v in json.loads(
    (project_root / 'configs' / 'label_schemes' / 'labelid2cluster_concr4.json').read_text()).items()}

CMAP_RANGE = {
    'pcc':     ('RdBu_r', -1.0, 1.0),
    'abs_pcc': ('hot',     0.0, 1.0),
    'im_pcc':  ('RdBu_r', -0.5, 0.5),
    'wpli':    ('YlOrRd',  0.0, 0.6),
    'plv':     ('YlOrRd',  0.0, 1.0),
}
COLORS_CLUSTER = ['#2C7BB6', '#D7191C', '#1A9641', '#FF7F00']

_PAT = re.compile(r'^P(\d+)_S(\d+)$')

def build_index(metric):
    root = project_root / 'data' / f'hypergraphs_pruned_{metric}'
    idx  = defaultdict(list)
    if not root.exists():
        log.warning(f'Directory non trovata: {root}')
        return idx
    for p in sorted(root.rglob('trial_*.pt')):
        m = _PAT.match(p.parent.name)
        if m:
            idx[int(m.group(1))].append(p)
    return idx

MAIN_IDX = build_index('abs_pcc')
ALL_SUBJ = sorted(MAIN_IDX.keys())
log.info(f'Soggetti trovati: {len(ALL_SUBJ)}')

## §2 — Calcolo Matrici per Soggetto × Cluster Semantico

Per ogni soggetto e ogni metrica: raggruppa i trial per cluster semantico (concr4)
e calcola la media delle `adj` per ciascun cluster → tensore `(n_subj, 4, 61, 61)`.

In [ ]:
SC_CACHE = CKPT_DIR / 'subject_cluster_conn.npz'

if SC_CACHE.exists():
    log.info(f'Cache trovata: {SC_CACHE}')
    _c = np.load(SC_CACHE, allow_pickle=True)
    CONN_SC  = _c['conn_sc'].item()    # metric → (n_subj, 4, 61, 61)
    SUBJ_IDS = _c['subj_ids'].tolist()
else:
    log.info('Calcolo matrici per soggetto × cluster semantico...')
    CONN_SC  = {m: [] for m in METRICS}
    SUBJ_IDS = []

    for sid in tqdm(ALL_SUBJ, desc='Soggetti'):
        paths_main = MAIN_IDX[sid]
        if len(paths_main) < 10:
            continue

        # accum[metric][sem_cluster] → lista di adj (61,61)
        accum = {m: {c: [] for c in range(N_SEM_CLUSTERS)} for m in METRICS}

        for metric in METRICS:
            idx_m   = build_index(metric)
            paths_m = idx_m.get(sid, [])
            for p in paths_m:
                try:
                    d = torch.load(p, weights_only=False)
                    adj = d['adj'].float().numpy()   # (61, 61)
                    y   = int(d['y'])                # word label
                    sc  = label2cluster.get(y, -1)   # semantic cluster
                    if sc < 0 or adj.shape != (N_CHANNELS, N_CHANNELS):
                        continue
                    accum[metric][sc].append(adj)
                except Exception:
                    continue

        # Soggetto valido se ha almeno 3 trial per ogni cluster semantico e metrica
        if any(len(accum[m][c]) < 3
               for m in METRICS for c in range(N_SEM_CLUSTERS)):
            continue

        for m in METRICS:
            # (4, 61, 61) — media per cluster semantico
            mat_sc = np.stack([np.mean(accum[m][c], axis=0)
                               for c in range(N_SEM_CLUSTERS)])
            CONN_SC[m].append(mat_sc)
        SUBJ_IDS.append(sid)

    for m in METRICS:
        CONN_SC[m] = np.stack(CONN_SC[m])   # (n_subj, 4, 61, 61)

    np.savez(SC_CACHE, conn_sc=CONN_SC, subj_ids=np.array(SUBJ_IDS))
    log.info(f'Salvato: {SC_CACHE}')

N_SUBJ = len(SUBJ_IDS)
log.info(f'Soggetti validi: {N_SUBJ}')
for m in METRICS:
    log.info(f'  {m}: {CONN_SC[m].shape}')   # (n_subj, 4, 61, 61)

## §3 — Grand Mean per Cluster Semantico

`GRAND_SC[metric][k]` = media su tutti i soggetti per il cluster semantico k → `(61,61)`

Rappresenta il pattern di connettività "tipico" della popolazione
quando immagina parole della categoria k.

In [ ]:
# GRAND_SC[metric] shape: (4, 61, 61)
GRAND_SC = {m: CONN_SC[m].mean(axis=0) for m in METRICS}

# Carica label nomi cluster semantici se disponibile
SEM_NAMES = {0: 'C0', 1: 'C1', 2: 'C2', 3: 'C3'}
try:
    label2idx = json.loads(
        (project_root / 'configs' / 'label_schemes' / 'label2idx.json').read_text())
    # Raggruppa parole per cluster
    words_by_cluster = defaultdict(list)
    for word, idx in label2idx.items():
        sc = label2cluster.get(idx, -1)
        if sc >= 0:
            words_by_cluster[sc].append(word)
    for c in range(N_SEM_CLUSTERS):
        sample = ', '.join(sorted(words_by_cluster[c])[:5])
        SEM_NAMES[c] = f'C{c} ({sample}...)'
        log.info(f'Cluster {c}: {len(words_by_cluster[c])} parole — es: {sample}')
except Exception as e:
    log.warning(f'label2idx non caricato: {e}')

# Visualizza grand mean per ogni cluster semantico e metrica
fig, axes = plt.subplots(len(METRICS), N_SEM_CLUSTERS,
                          figsize=(3.5 * N_SEM_CLUSTERS, 3.5 * len(METRICS)))
fig.suptitle(f'Grand Mean per Cluster Semantico (concr4) — tutti i soggetti (n={N_SUBJ})',
             fontsize=13, fontweight='bold')

for row_i, metric in enumerate(METRICS):
    cmap, vmin, vmax = CMAP_RANGE[metric]
    for c in range(N_SEM_CLUSTERS):
        ax = axes[row_i][c]
        im = ax.imshow(GRAND_SC[metric][c], cmap=cmap, vmin=vmin, vmax=vmax,
                       interpolation='nearest')
        if row_i == 0:
            ax.set_title(SEM_NAMES[c], fontsize=9,
                         color=COLORS_CLUSTER[c], fontweight='bold')
        if c == 0:
            ax.set_ylabel(metric, fontsize=9)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg17_grand_mean_per_sem_cluster.png', dpi=150, bbox_inches='tight')
plt.show()
log.info('Grand mean per cluster semantico salvato')

## §4 — Confronto tra Cluster Semantici (diff inter-cluster)

Per ogni metrica e ogni coppia di cluster semantici: `grand_C_i − grand_C_j`
Mostra dove la connettività cambia sistematicamente tra categorie di parole.

In [ ]:
import itertools

pairs = list(itertools.combinations(range(N_SEM_CLUSTERS), 2))  # (0,1),(0,2),(0,3),(1,2)...

for metric in [PRIMARY]:   # espandi a METRICS per tutte le metriche
    fig, axes = plt.subplots(1, len(pairs), figsize=(3.5 * len(pairs), 3.8))
    fig.suptitle(f'{metric} — Grand mean Ci − Grand mean Cj (inter-cluster diff)',
                 fontsize=11, fontweight='bold')

    for ax, (ci, cj) in zip(axes, pairs):
        diff = GRAND_SC[metric][ci] - GRAND_SC[metric][cj]
        d_clamp = np.abs(diff).max() * 0.9
        im = ax.imshow(diff, cmap='RdBu_r', vmin=-d_clamp, vmax=d_clamp,
                       interpolation='nearest')
        ax.set_title(f'C{ci} − C{cj}', fontsize=10, fontweight='bold')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    plt.tight_layout()
    plt.savefig(FIG_DIR / f'eeg17_{metric}_intercluster_diff.png', dpi=150, bbox_inches='tight')
    plt.show()
log.info('Inter-cluster diff salvato')

## §5 — Soggetto vs Grand Mean per Cluster Semantico

Per ogni soggetto e ogni cluster semantico k:
`diff[soggetto, k] = CONN_SC[soggetto, k] − GRAND_SC[k]`

Risponde a: quando il soggetto X immagina parole categoria k,
quanto differisce dal pattern medio della popolazione per la stessa categoria?

In [ ]:
# Carica accuracy per etichettare i soggetti
acc_data = {}
for label, fpath in [('EEG_12', ACC_FILE_12), ('EEG_13', ACC_FILE_13)]:
    if fpath.exists():
        df = pd.read_csv(fpath)
        if 'Subject' in df.columns:
            df['sid'] = df['Subject'].str.extract(r'(\d+)').astype(int)
        col = 'Test bAcc' if 'Test bAcc' in df.columns else 'test_bacc'
        acc_data[label] = dict(zip(df['sid'], df[col]))
        log.info(f'Caricato {label}: {len(acc_data[label])} soggetti')
    else:
        log.warning(f'{label} non trovato')

acc12 = acc_data.get('EEG_12', {})
acc13 = acc_data.get('EEG_13', {})

metric = PRIMARY
cmap, vmin, vmax = CMAP_RANGE[metric]

# Range diff globale (stesso per tutti i cluster semantici)
all_diffs = CONN_SC[metric] - GRAND_SC[metric][None, :, :, :]   # (n_subj, 4, 61, 61)
d_clamp   = min(float(np.abs(all_diffs).max() * 0.8), 0.4)

for c in range(N_SEM_CLUSTERS):
    grand_c = GRAND_SC[metric][c]   # (61, 61)

    ncols = min(8, N_SUBJ + 1)
    nrows = math.ceil((N_SUBJ + 1) / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(2.8 * ncols, 2.8 * nrows + 0.8))
    fig.suptitle(
        f'{metric} | Cluster semantico {c} — soggetto − grand mean C{c}\n'
        f'colormap RdBu_r ±{d_clamp:.2f}  |  titolo = acc12 | acc13',
        fontsize=10, fontweight='bold', color=COLORS_CLUSTER[c]
    )
    axes_flat = np.array(axes).flatten()

    # Pannello 0: grand mean cluster c come riferimento
    im_ref = axes_flat[0].imshow(grand_c, cmap=cmap, vmin=vmin, vmax=vmax,
                                  interpolation='nearest')
    axes_flat[0].set_title(f'Grand mean\nC{c}', fontsize=8,
                            fontweight='bold', color=COLORS_CLUSTER[c])
    axes_flat[0].axis('off')
    plt.colorbar(im_ref, ax=axes_flat[0], fraction=0.05, pad=0.02)

    # Ordina soggetti per accuracy decrescente (EEG_13 o EEG_12)
    acc_sort = acc13 if acc13 else acc12
    order = sorted(range(N_SUBJ),
                   key=lambda i: acc_sort.get(SUBJ_IDS[i], 0), reverse=True)

    for plot_i, subj_idx in enumerate(order, start=1):
        sid  = SUBJ_IDS[subj_idx]
        diff = CONN_SC[metric][subj_idx, c] - grand_c
        ax   = axes_flat[plot_i]
        ax.imshow(diff, cmap='RdBu_r', vmin=-d_clamp, vmax=d_clamp,
                  interpolation='nearest')
        a12 = acc12.get(sid, np.nan); a13 = acc13.get(sid, np.nan)
        ax.set_title(f'P{sid:03d}\n{a12:.2f}|{a13:.2f}', fontsize=7,
                     color='black', fontweight='bold')
        ax.axis('off')

    for i in range(N_SUBJ + 1, len(axes_flat)):
        axes_flat[i].axis('off')

    plt.tight_layout()
    fname = FIG_DIR / f'eeg17_{metric}_sem_cluster{c}_subj_vs_grand.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    log.info(f'Cluster semantico {c} → {fname.name}')

## §6 — Deviazione Media per Soggetto × Cluster Semantico

`dev[soggetto, k]` = deviazione media assoluta dal grand mean del cluster k.

Produce una matrice `(n_subj, 4)` — soggetti ordinati per accuracy,
cluster semantici in colonna. Mostra se soggetti proficient deviino
di più o di meno dalla media per categorie specifiche.

In [ ]:
metric = PRIMARY

# dev[i, k] = mean |CONN_SC[i,k] - GRAND_SC[k]|  scalar per (soggetto, cluster_sem)
diffs_all = CONN_SC[metric] - GRAND_SC[metric][None, :, :, :]   # (n_subj, 4, 61, 61)
DEV = np.abs(diffs_all).mean(axis=(2, 3))   # (n_subj, 4)

# Ordina soggetti per accuracy EEG_13 decrescente
acc_sort = acc13 if acc13 else acc12
order = sorted(range(N_SUBJ),
               key=lambda i: acc_sort.get(SUBJ_IDS[i], 0), reverse=True)

DEV_sorted    = DEV[order]
SUBJ_sorted   = [SUBJ_IDS[i] for i in order]
accs_sorted   = np.array([acc_sort.get(sid, np.nan) for sid in SUBJ_sorted])

fig, axes = plt.subplots(1, 2, figsize=(16, max(6, N_SUBJ * 0.18 + 1)))
fig.suptitle(f'{metric} — Deviazione soggetto dal grand mean per cluster semantico\n'
             f'(soggetti ordinati per accuracy EEG_13 decrescente)',
             fontsize=12, fontweight='bold')

# ── Heatmap deviazione ────────────────────────────────────────────────────────
ax = axes[0]
im = ax.imshow(DEV_sorted, cmap='YlOrRd', aspect='auto',
               vmin=DEV_sorted.min(), vmax=DEV_sorted.max())
ax.set_xticks(range(N_SEM_CLUSTERS))
ax.set_xticklabels([f'C{c}' for c in range(N_SEM_CLUSTERS)], fontweight='bold')
ax.set_yticks(range(N_SUBJ))
ax.set_yticklabels([f'P{sid:03d}' for sid in SUBJ_sorted], fontsize=6)
ax.set_xlabel('Cluster semantico'); ax.set_ylabel('Soggetto (→ acc decrescente)')
ax.set_title('|soggetto − grand_mean| per cluster')
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

# ── Scatter: deviazione media vs accuracy ─────────────────────────────────────
ax2 = axes[1]
mean_dev = DEV.mean(axis=1)   # media su tutti e 4 i cluster semantici
accs_all  = np.array([acc_sort.get(SUBJ_IDS[i], np.nan) for i in range(N_SUBJ)])
valid     = ~np.isnan(accs_all)
ax2.scatter(mean_dev[valid], accs_all[valid], alpha=0.7, s=60, color='steelblue')
for i in np.where(valid)[0]:
    ax2.annotate(f'P{SUBJ_IDS[i]:03d}', (mean_dev[i], accs_all[i]),
                 fontsize=5, xytext=(2, 2), textcoords='offset points')

# Regressione lineare
from numpy.polynomial import polynomial as P
coefs = np.polyfit(mean_dev[valid], accs_all[valid], 1)
x_line = np.linspace(mean_dev[valid].min(), mean_dev[valid].max(), 100)
ax2.plot(x_line, np.polyval(coefs, x_line), 'r--', linewidth=1.5,
         label=f'y={coefs[0]:.3f}x+{coefs[1]:.3f}')
corr = np.corrcoef(mean_dev[valid], accs_all[valid])[0, 1]
ax2.set_xlabel('Deviazione media dal grand mean (tutti i cluster)')
ax2.set_ylabel('bAcc')
ax2.set_title(f'Deviazione vs Accuracy  r={corr:.3f}')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / f'eeg17_{metric}_deviation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
log.info(f'Correlazione deviazione-accuracy: r={corr:.3f}')

# Stampa tabella
df_dev = pd.DataFrame(DEV_sorted,
                       columns=[f'dev_C{c}' for c in range(N_SEM_CLUSTERS)])
df_dev.insert(0, 'Subject', [f'P{s:03d}' for s in SUBJ_sorted])
df_dev['acc_EEG_13'] = accs_sorted
df_dev['dev_mean']   = DEV_sorted.mean(axis=1)
df_dev.to_csv(FIG_DIR / 'eeg17_subject_deviation_per_semcluster.csv', index=False)
print(df_dev.to_string(index=False, float_format='{:.4f}'.format))